In [1]:
import pandas as pd
import numpy as np
from unidecode import unidecode
pd.set_option("display.max_columns", None)


df_aux = pd.read_excel("/lakehouse/default/Files/obras/tb_aux_etapas_consideradas.xlsx")
df_aux.columns = df_aux.columns.str.lower().str.replace("ç", "c").str.replace(" ", "_")

df_aux_etapas_consideradas = df_aux.loc[df_aux["etapa_atual"].notna()].copy()


df_os = pd.read_csv("/lakehouse/default/Files/obras/os_servicos_obras.csv", sep=";")

df_os.columns = (
    df_os.columns.str.lower()
    .str.replace(r"[()/:º-]", "", regex=True)
    .str.replace("ç", "c")
    .str.replace("m²", "m2")
    .str.strip()
    .str.replace(r"\s+", "_", regex=True)
)
df_os.columns = [unidecode(col) for col in df_os.columns]
df_os['data_de_solicitacao'] = pd.to_datetime(df_os['data_de_solicitacao'], dayfirst=True)

df_os['servico'] = df_os['servico'].str.upper()

df_alvaras = pd.merge(
    df_os,
    df_aux_etapas_consideradas,
    on=["servico", "etapa_atual"],
    how="outer",
    indicator=True,
)

df_alvaras['data_de_solicitacao'].max()

StatementMeta(, 87a7525d-403d-4aaf-aeac-f5072a864c57, 3, Finished, Available, Finished, False)

Timestamp('2026-04-19 16:54:00')

In [2]:
servicos_interesse = [
    "ALVARÁ DE CONSTRUÇÃO",
    "ALVARÁ DE DEMOLIÇÃO",
    "ALVARÁ DE INSTALAÇÃO DE ANTENAS",
    "1 - ALVARÁ INSTALAÇÃO DE ELEVADORES E ESTEIRAS ROLANTE",
    "ALVARÁ DE MURO DE ARRIMO",
    "2 - ALVARÁ OPERAÇÃO DE ELEVADORES E ESTEIRAS ROLANTE",
    "2 - CERTIFICADO DE CONCLUSÃO ETR", # Alvará de Operação/Conclusão de Antenas 
    "3 - AUTO DE REGULARIZAÇÃO ETR", # Alvará de regularização de antenas
    "ALVARÁ DE REFORMA",
    "ALVARÁ DE REGULARIZAÇÃO",
    "ALVARÁ DE PROJETO MODIFICADO",
    "ALVARÁ DE EXECUÇÃO", #"EXECUÇÃO DE PROJETO (ALVARÁ DE EXECUÇÃO)",
    "ALVARÁ DE REGULARIZAÇÃO DE ANTENAS (AUTO DE REGULARIZAÇÃO ETR)",
    "ALVARÁ DE APROVAÇÃO DE PROJETO", #"APROVAÇÃO DE PROJETO",
    "ALVARÁ DE TERRAPLENAGEM",
    "CERTIDÃO DE ANUÊNCIA DESDOBRO DE ÁREA MAIOR",
    "CERTIDÃO DE DEMOLIÇÃO",
    "CERTIDÃO DE DESDOBRO OU FUSÃO",
    "CERTIDÃO DE USO DO SOLO",
    "CERTIDÃO INFORMATIVA",
    "CERTIDÃO REVALIDAÇÃO",
    "HABITE-SE",
    "INSCRIÇÃO E RENOVAÇÃO PROFISSIONAL",
    "OBRAS PÚBLICAS",
    "ALVARÁ DE OPERAÇÃO/CONCLUSÃO DE ANTENAS",
]
df_alvaras = df_alvaras.loc[df_alvaras["servico"].isin(servicos_interesse)].copy()
df_alvaras['data_de_solicitacao'].max()

StatementMeta(, 87a7525d-403d-4aaf-aeac-f5072a864c57, 4, Finished, Available, Finished, False)

Timestamp('2026-04-19 16:54:00')

In [3]:



df_alvaras['status_alvara'] = np.where(
    df_alvaras['_merge'] == 'both',
    'expedido',
    'em_tramite'
)
colunas_totalmente_nulas = (
    df_alvaras
    .isna()
    .mean()
    .to_frame("percentual_nulos")
    .query("percentual_nulos == 1")
    .index.tolist()
)

df_alvaras = df_alvaras.drop(columns=colunas_totalmente_nulas + ["_merge"])
df_alvaras['n_da_solicitacao'] = df_alvaras['n_da_solicitacao'].fillna(999999).astype(int)

StatementMeta(, 87a7525d-403d-4aaf-aeac-f5072a864c57, 5, Finished, Available, Finished, False)

In [4]:
# análise:
df_alvaras_status = df_alvaras.groupby(['servico', 'status_alvara'], as_index=False).size()
df_alvaras_status = df_alvaras_status.pivot(
    index="servico",
    columns="status_alvara",
    values="size"
).reset_index().sort_values("em_tramite", ascending=False)

df_alvaras_status['total'] = df_alvaras_status['em_tramite'] + df_alvaras_status['expedido']
df_alvaras_status['pct_expedido'] = df_alvaras_status['expedido'] / df_alvaras_status['total']

df_alvaras_status[[
    "em_tramite", "expedido", "total", "pct_expedido"
]] = df_alvaras_status[[
    "em_tramite", "expedido", "total", "pct_expedido"
]].fillna(0)
df_alvaras_status

StatementMeta(, 87a7525d-403d-4aaf-aeac-f5072a864c57, 6, Finished, Available, Finished, False)

status_alvara,servico,em_tramite,expedido,total,pct_expedido
12,ALVARÁ DE REGULARIZAÇÃO,1211.0,110.0,1321.0,0.083270
17,CERTIDÃO DE USO DO SOLO,1119.0,453.0,1572.0,0.288168
5,ALVARÁ DE CONSTRUÇÃO,710.0,56.0,766.0,0.073107
16,CERTIDÃO DE DESDOBRO OU FUSÃO,625.0,159.0,784.0,0.202806
18,CERTIDÃO INFORMATIVA,427.0,0.0,0.0,0.000000
21,INSCRIÇÃO E RENOVAÇÃO PROFISSIONAL,383.0,576.0,959.0,0.600626
6,ALVARÁ DE DEMOLIÇÃO,280.0,206.0,486.0,0.423868
20,HABITE-SE,237.0,8.0,245.0,0.032653
15,CERTIDÃO DE DEMOLIÇÃO,220.0,22.0,242.0,0.090909
11,ALVARÁ DE REFORMA,219.0,22.0,241.0,0.091286


In [5]:
# escrita no banco
sdf = spark.createDataFrame(df_alvaras)
(
    sdf
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_alvaras_obras")
)

StatementMeta(, 87a7525d-403d-4aaf-aeac-f5072a864c57, 7, Finished, Available, Finished, True)